In [3]:
# Cell 1 - Import library dan konfigurasi utama

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

warnings.filterwarnings("ignore")


# ============================================================
# MAIN CONFIG - MUDAH DIEDIT
# ============================================================

# Base input framed dataset
FRAMED_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/Framed Dataset")

DEV_DIR = FRAMED_BASE_DIR / "Dataset Development"
TEST_DIR = FRAMED_BASE_DIR / "Dataset Testing"

# Output preprocessing
OUT_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/Preprocessing") / "Preprocessed_GC25_Z008"

# Ground correction angle, editable
GC_ANGLE_DEG = 25.0

# Z threshold ground removal, editable
Z_GROUND_THRESHOLD = 0.08  # meter, 0.08 m = 8 cm

# ROI after correction, fixed sesuai keputusan
ROI_X_MIN, ROI_X_MAX = 0.0, 3.0
ROI_Y_MIN, ROI_Y_MAX = -1.5, 1.5
ROI_Z_MIN, ROI_Z_MAX = 0.0, 2.0

# Floor leveling offset percentile
# Dipakai per frame setelah rotasi untuk membuat lantai mendekati Z = 0
FLOOR_OFFSET_PERCENTILE = 1.0

# Low-Z residual zone untuk evaluasi sebelum DBSCAN
# Setelah threshold 0.08, zona rendah yang masih tersisa misalnya 0.08 - 0.15 m.
LOW_Z_RESIDUAL_LIMIT = 0.15

# Dataset information
ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

DEV_SUBJECTS = [
    "Adelia",
    "Afi",
    "Aswangga",
    "Bustan",
    "Dilia",
    "Eldivo",
    "Fathir",
    "Lina",
    "Manda",
    "Miftah",
    "Teguh",
    "Tsamara",
]

TEST_ROOMS = ["Controlled Room", "Uncontrolled Room"]

TEST_SUBJECTS = [
    "Kanaya",
    "Naila",
    "Nana",
    "Rega",
    "Zaira",
]

FILE_IDS = list(range(1, 10))  # 1.csv sampai 9.csv

# Required columns
REQUIRED_COLUMNS = ["frame_id", "Timestamp", "X", "Y", "Z", "Reflectivity"]

# Output behavior
OVERWRITE_OUTPUT = False  # ini nya matiin

# CSV save precision
FLOAT_FORMAT = "%.6f"

print("===== CONFIG SUMMARY =====")
print(f"Input base        : {FRAMED_BASE_DIR}")
print(f"Output base       : {OUT_BASE_DIR}")
print(f"GC angle          : {GC_ANGLE_DEG} degree")
print(f"Z threshold       : {Z_GROUND_THRESHOLD} m")
print(f"ROI X             : {ROI_X_MIN} to {ROI_X_MAX}")
print(f"ROI Y             : {ROI_Y_MIN} to {ROI_Y_MAX}")
print(f"ROI Z             : {ROI_Z_MIN} to {ROI_Z_MAX}")
print(f"Low-Z residual <= : {LOW_Z_RESIDUAL_LIMIT} m")

===== CONFIG SUMMARY =====
Input base        : /media/spell/Spell-lab/Lidar/Framed Dataset
Output base       : /media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008
GC angle          : 25.0 degree
Z threshold       : 0.08 m
ROI X             : 0.0 to 3.0
ROI Y             : -1.5 to 1.5
ROI Z             : 0.0 to 2.0
Low-Z residual <= : 0.15 m


In [4]:
# Cell 2 - Membuat struktur folder output

GC25_ROI_DIR = OUT_BASE_DIR / "GC25_ROI"
GC25_Z_DIR = OUT_BASE_DIR / f"GC25_Z{int(Z_GROUND_THRESHOLD * 1000):03d}"

METRICS_DIR = OUT_BASE_DIR / "_metrics"
LOG_DIR = OUT_BASE_DIR / "_logs"

for d in [OUT_BASE_DIR, GC25_ROI_DIR, GC25_Z_DIR, METRICS_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Simpan konfigurasi eksperimen
config = {
    "framed_base_dir": str(FRAMED_BASE_DIR),
    "out_base_dir": str(OUT_BASE_DIR),
    "gc_angle_deg": GC_ANGLE_DEG,
    "z_ground_threshold_m": Z_GROUND_THRESHOLD,
    "roi": {
        "x_min": ROI_X_MIN,
        "x_max": ROI_X_MAX,
        "y_min": ROI_Y_MIN,
        "y_max": ROI_Y_MAX,
        "z_min": ROI_Z_MIN,
        "z_max": ROI_Z_MAX,
    },
    "floor_offset_percentile": FLOOR_OFFSET_PERCENTILE,
    "low_z_residual_limit_m": LOW_Z_RESIDUAL_LIMIT,
    "activities": ACTIVITIES,
    "dev_subjects": DEV_SUBJECTS,
    "test_rooms": TEST_ROOMS,
    "test_subjects": TEST_SUBJECTS,
    "file_ids": FILE_IDS,
}

config_path = OUT_BASE_DIR / "preprocessing_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Output folders created.")
print(f"GC25_ROI_DIR : {GC25_ROI_DIR}")
print(f"GC25_Z_DIR   : {GC25_Z_DIR}")
print(f"METRICS_DIR  : {METRICS_DIR}")
print(f"Config saved : {config_path}")

Output folders created.
GC25_ROI_DIR : /media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008/GC25_ROI
GC25_Z_DIR   : /media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008/GC25_Z080
METRICS_DIR  : /media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008/_metrics
Config saved : /media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008/preprocessing_config.json


In [5]:
# Cell 3 - Membuat manifest seluruh file dataset

def build_dataset_manifest():
    records = []

    # ----------------------------
    # Dataset Development
    # ----------------------------
    for activity in ACTIVITIES:
        for subject in DEV_SUBJECTS:
            for file_id in FILE_IDS:
                csv_path = DEV_DIR / activity / subject / f"{file_id}.csv"
                records.append({
                    "split": "development",
                    "room": "development",
                    "activity": activity,
                    "subject": subject,
                    "file_id": file_id,
                    "csv_path": csv_path,
                    "exists": csv_path.exists(),
                })

    # ----------------------------
    # Dataset Testing
    # Structure:
    # Dataset Testing / Controlled Room / Activity / Subject / 1.csv
    # Dataset Testing / Uncontrolled Room / Activity / Subject / 1.csv
    # ----------------------------
    for room in TEST_ROOMS:
        for activity in ACTIVITIES:
            for subject in TEST_SUBJECTS:
                for file_id in FILE_IDS:
                    csv_path = TEST_DIR / room / activity / subject / f"{file_id}.csv"
                    records.append({
                        "split": "testing",
                        "room": room,
                        "activity": activity,
                        "subject": subject,
                        "file_id": file_id,
                        "csv_path": csv_path,
                        "exists": csv_path.exists(),
                    })

    manifest_df = pd.DataFrame(records)
    return manifest_df


manifest_df = build_dataset_manifest()

manifest_path = METRICS_DIR / "dataset_manifest.csv"
manifest_df.assign(csv_path=manifest_df["csv_path"].astype(str)).to_csv(manifest_path, index=False)

print("===== DATASET MANIFEST =====")
print(f"Total expected files : {len(manifest_df):,}")
print(f"Existing files       : {manifest_df['exists'].sum():,}")
print(f"Missing files        : {(~manifest_df['exists']).sum():,}")
print(f"Manifest saved       : {manifest_path}")

display(manifest_df.head())

missing_df = manifest_df[~manifest_df["exists"]].copy()
if len(missing_df) > 0:
    missing_path = LOG_DIR / "missing_files.csv"
    missing_df.assign(csv_path=missing_df["csv_path"].astype(str)).to_csv(missing_path, index=False)
    print(f"Missing file report saved: {missing_path}")

===== DATASET MANIFEST =====
Total expected files : 792
Existing files       : 792
Missing files        : 0
Manifest saved       : /media/spell/Spell-lab/Lidar/Preprocessing/Preprocessed_GC25_Z008/_metrics/dataset_manifest.csv


,split,room,activity,subject,file_id,csv_path,exists
0,development,development,Bungkuk,Adelia,1,/media/spell/Spell-lab/Lidar/Framed Dataset/Da...,True
1,development,development,Bungkuk,Adelia,2,/media/spell/Spell-lab/Lidar/Framed Dataset/Da...,True
2,development,development,Bungkuk,Adelia,3,/media/spell/Spell-lab/Lidar/Framed Dataset/Da...,True
3,development,development,Bungkuk,Adelia,4,/media/spell/Spell-lab/Lidar/Framed Dataset/Da...,True
4,development,development,Bungkuk,Adelia,5,/media/spell/Spell-lab/Lidar/Framed Dataset/Da...,True


In [6]:
# Cell 4 - Fungsi rotasi, leveling, ROI, dan validasi CSV

def validate_columns(df: pd.DataFrame, csv_path: Path):
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {csv_path}: {missing_cols}")


def clean_numeric_rows(df: pd.DataFrame):
    """
    Membersihkan baris yang punya NaN/Inf pada kolom numerik utama.
    """
    numeric_cols = ["X", "Y", "Z", "Reflectivity"]

    df = df.copy()
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    valid_mask = np.isfinite(df[numeric_cols].to_numpy()).all(axis=1)
    return df.loc[valid_mask].copy()


def rotate_y_numpy(x, y, z, angle_deg: float):
    """
    Rotasi terhadap sumbu Y.
    
    Rumus:
    X_corr = X cos(theta) + Z sin(theta)
    Y_corr = Y
    Z_corr = -X sin(theta) + Z cos(theta)
    """
    theta = np.deg2rad(angle_deg)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    x_corr = x * cos_t + z * sin_t
    y_corr = y
    z_corr = -x * sin_t + z * cos_t

    return x_corr, y_corr, z_corr


def get_roi_mask_after_correction(df: pd.DataFrame):
    """
    ROI setelah ground correction dan leveling.
    """
    return (
        (df["X_corr"] >= ROI_X_MIN) & (df["X_corr"] <= ROI_X_MAX) &
        (df["Y_corr"] >= ROI_Y_MIN) & (df["Y_corr"] <= ROI_Y_MAX) &
        (df["Z_level"] >= ROI_Z_MIN) & (df["Z_level"] <= ROI_Z_MAX)
    )


def apply_z_ground_threshold(df_roi: pd.DataFrame, z_threshold: float):
    """
    Ground removal:
    keep jika Z_level > threshold.
    remove jika Z_level <= threshold.
    """
    return df_roi[df_roi["Z_level"] > z_threshold].copy()


def safe_range(series: pd.Series):
    if len(series) == 0:
        return np.nan, np.nan, np.nan
    vmin = float(series.min())
    vmax = float(series.max())
    return vmin, vmax, vmax - vmin


def safe_percentile(values, q):
    if len(values) == 0:
        return np.nan
    return float(np.percentile(values, q))

In [7]:
# Cell 5 - Fungsi metrik evaluasi sebelum DBSCAN

def compute_height_distribution(prefix: str, df: pd.DataFrame, z_col: str = "Z_level"):
    """
    Menghitung ringkasan distribusi tinggi Z.
    Dipakai untuk mengevaluasi height distribution sebelum DBSCAN.
    """
    if len(df) == 0:
        return {
            f"{prefix}_z_min": np.nan,
            f"{prefix}_z_max": np.nan,
            f"{prefix}_z_mean": np.nan,
            f"{prefix}_z_std": np.nan,
            f"{prefix}_z_p01": np.nan,
            f"{prefix}_z_p05": np.nan,
            f"{prefix}_z_p25": np.nan,
            f"{prefix}_z_p50": np.nan,
            f"{prefix}_z_p75": np.nan,
            f"{prefix}_z_p95": np.nan,
            f"{prefix}_z_p99": np.nan,
        }

    z = df[z_col].to_numpy()

    return {
        f"{prefix}_z_min": float(np.min(z)),
        f"{prefix}_z_max": float(np.max(z)),
        f"{prefix}_z_mean": float(np.mean(z)),
        f"{prefix}_z_std": float(np.std(z)),
        f"{prefix}_z_p01": safe_percentile(z, 1),
        f"{prefix}_z_p05": safe_percentile(z, 5),
        f"{prefix}_z_p25": safe_percentile(z, 25),
        f"{prefix}_z_p50": safe_percentile(z, 50),
        f"{prefix}_z_p75": safe_percentile(z, 75),
        f"{prefix}_z_p95": safe_percentile(z, 95),
        f"{prefix}_z_p99": safe_percentile(z, 99),
    }


def compute_low_z_spread(df_after_z: pd.DataFrame):
    """
    Evaluasi titik rendah yang masih tersisa setelah Z-threshold.

    Low-Z residual didefinisikan sebagai:
    Z_ground_threshold < Z_level <= LOW_Z_RESIDUAL_LIMIT

    Kenapa bukan Z_level <= threshold?
    Karena titik <= threshold sudah dibuang.
    Jadi yang dicek adalah sisa titik rendah di atas threshold.
    """
    if len(df_after_z) == 0:
        return {
            "low_z_residual_count": 0,
            "low_z_residual_ratio_after_z": np.nan,
            "low_z_x_min": np.nan,
            "low_z_x_max": np.nan,
            "low_z_x_range": np.nan,
            "low_z_y_min": np.nan,
            "low_z_y_max": np.nan,
            "low_z_y_range": np.nan,
        }

    low_mask = df_after_z["Z_level"] <= LOW_Z_RESIDUAL_LIMIT
    low_df = df_after_z.loc[low_mask]

    low_count = int(len(low_df))
    low_ratio = low_count / len(df_after_z) if len(df_after_z) > 0 else np.nan

    x_min, x_max, x_range = safe_range(low_df["X_corr"]) if low_count > 0 else (np.nan, np.nan, np.nan)
    y_min, y_max, y_range = safe_range(low_df["Y_corr"]) if low_count > 0 else (np.nan, np.nan, np.nan)

    return {
        "low_z_residual_count": low_count,
        "low_z_residual_ratio_after_z": low_ratio,
        "low_z_x_min": x_min,
        "low_z_x_max": x_max,
        "low_z_x_range": x_range,
        "low_z_y_min": y_min,
        "low_z_y_max": y_max,
        "low_z_y_range": y_range,
    }


def compute_pre_dbscan_metrics(
    metadata: dict,
    frame_id,
    points_raw: int,
    points_clean: int,
    floor_offset: float,
    df_roi: pd.DataFrame,
    df_after_z: pd.DataFrame,
):
    """
    Menghitung metrik per frame sebelum DBSCAN.
    """
    points_after_roi = len(df_roi)
    points_after_z = len(df_after_z)

    roi_retained_ratio = points_after_roi / points_clean if points_clean > 0 else np.nan
    z_retained_ratio_from_roi = points_after_z / points_after_roi if points_after_roi > 0 else np.nan
    z_removed_ratio_from_roi = 1.0 - z_retained_ratio_from_roi if not np.isnan(z_retained_ratio_from_roi) else np.nan

    # Retained points ratio utama setelah threshold
    retained_points_ratio = points_after_z / points_after_roi if points_after_roi > 0 else np.nan

    # Low-Z residual sebelum threshold dalam ROI
    low_z_before_count = int((df_roi["Z_level"] <= LOW_Z_RESIDUAL_LIMIT).sum()) if points_after_roi > 0 else 0
    low_z_before_ratio = low_z_before_count / points_after_roi if points_after_roi > 0 else np.nan

    # Low-Z residual setelah threshold
    low_z_spread = compute_low_z_spread(df_after_z)

    metrics = {
        **metadata,
        "frame_id": frame_id,
        "gc_angle_deg": GC_ANGLE_DEG,
        "z_ground_threshold": Z_GROUND_THRESHOLD,
        "floor_offset": floor_offset,

        "points_raw": points_raw,
        "points_clean": points_clean,
        "points_after_roi": points_after_roi,
        "points_after_z_threshold": points_after_z,

        "roi_retained_ratio": roi_retained_ratio,
        "retained_points_ratio": retained_points_ratio,
        "z_retained_ratio_from_roi": z_retained_ratio_from_roi,
        "z_removed_ratio_from_roi": z_removed_ratio_from_roi,

        "low_z_before_count": low_z_before_count,
        "low_z_before_ratio": low_z_before_ratio,
    }

    # Height distribution sebelum dan setelah Z threshold
    metrics.update(compute_height_distribution("roi", df_roi))
    metrics.update(compute_height_distribution("after_z", df_after_z))

    # XY spread titik low-Z residual setelah threshold
    metrics.update(low_z_spread)

    return metrics

In [8]:
# Cell 6 - Fungsi proses satu file CSV

OUTPUT_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]


def process_one_csv_file(row: pd.Series):
    """
    Proses satu CSV:
    1. Load raw framed CSV
    2. Validasi kolom
    3. Proses per frame:
       - rotasi Y sebesar GC_ANGLE_DEG
       - floor offset percentile per frame
       - Z_level = Z_corr - floor_offset
       - ROI filtering
       - Z-threshold ground removal
       - hitung metrik sebelum DBSCAN
    4. Simpan:
       - GC25_ROI
       - GC25_Z008
       - frame metrics
    """
    csv_path = Path(row["csv_path"])

    metadata = {
        "split": row["split"],
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "file_id": int(row["file_id"]),
        "source_csv": str(csv_path),
    }

    if not csv_path.exists():
        return [], {
            **metadata,
            "status": "missing_file",
            "error_message": "CSV file does not exist",
        }

    # Relative path mengikuti struktur asli dari FRAMED_BASE_DIR
    rel_path = csv_path.relative_to(FRAMED_BASE_DIR)

    out_roi_path = GC25_ROI_DIR / rel_path
    out_z_path = GC25_Z_DIR / rel_path

    out_roi_path.parent.mkdir(parents=True, exist_ok=True)
    out_z_path.parent.mkdir(parents=True, exist_ok=True)

    if (not OVERWRITE_OUTPUT) and out_roi_path.exists() and out_z_path.exists():
        return [], {
            **metadata,
            "status": "skipped_exists",
            "error_message": "",
        }

    try:
        df = pd.read_csv(csv_path)
        validate_columns(df, csv_path)

        points_raw_file = len(df)
        df_clean = clean_numeric_rows(df)
        points_clean_file = len(df_clean)

        if points_clean_file == 0:
            # Simpan file kosong
            pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(out_roi_path, index=False)
            pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(out_z_path, index=False)

            return [], {
                **metadata,
                "status": "empty_after_cleaning",
                "points_raw_file": points_raw_file,
                "points_clean_file": points_clean_file,
                "points_roi_file": 0,
                "points_z_file": 0,
                "error_message": "",
            }

        roi_parts = []
        z_parts = []
        frame_metrics = []

        # Proses per frame agar floor_offset adaptif per frame
        for frame_id, frame_df in df_clean.groupby("frame_id", sort=True):
            frame_df = frame_df.copy()
            points_raw_frame = int((df["frame_id"] == frame_id).sum())
            points_clean_frame = len(frame_df)

            x = frame_df["X"].to_numpy(dtype=float)
            y = frame_df["Y"].to_numpy(dtype=float)
            z = frame_df["Z"].to_numpy(dtype=float)

            x_corr, y_corr, z_corr = rotate_y_numpy(x, y, z, GC_ANGLE_DEG)

            frame_df["X_corr"] = x_corr
            frame_df["Y_corr"] = y_corr
            frame_df["Z_corr"] = z_corr

            # Floor offset per frame
            floor_offset = safe_percentile(frame_df["Z_corr"].to_numpy(), FLOOR_OFFSET_PERCENTILE)
            frame_df["Z_level"] = frame_df["Z_corr"] - floor_offset

            # ROI after correction
            roi_mask = get_roi_mask_after_correction(frame_df)
            frame_roi = frame_df.loc[roi_mask, OUTPUT_COLUMNS].copy()

            # Z threshold ground removal
            frame_after_z = apply_z_ground_threshold(frame_roi, Z_GROUND_THRESHOLD)

            if len(frame_roi) > 0:
                roi_parts.append(frame_roi)

            if len(frame_after_z) > 0:
                z_parts.append(frame_after_z)

            metrics = compute_pre_dbscan_metrics(
                metadata=metadata,
                frame_id=frame_id,
                points_raw=points_raw_frame,
                points_clean=points_clean_frame,
                floor_offset=floor_offset,
                df_roi=frame_roi,
                df_after_z=frame_after_z,
            )
            frame_metrics.append(metrics)

        # Gabungkan output file
        if len(roi_parts) > 0:
            out_roi_df = pd.concat(roi_parts, ignore_index=True)
        else:
            out_roi_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

        if len(z_parts) > 0:
            out_z_df = pd.concat(z_parts, ignore_index=True)
        else:
            out_z_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

        # Save output CSV
        out_roi_df.to_csv(out_roi_path, index=False, float_format=FLOAT_FORMAT)
        out_z_df.to_csv(out_z_path, index=False, float_format=FLOAT_FORMAT)

        file_log = {
            **metadata,
            "status": "success",
            "points_raw_file": points_raw_file,
            "points_clean_file": points_clean_file,
            "points_roi_file": len(out_roi_df),
            "points_z_file": len(out_z_df),
            "roi_file_retained_ratio": len(out_roi_df) / points_clean_file if points_clean_file > 0 else np.nan,
            "z_file_retained_ratio_from_roi": len(out_z_df) / len(out_roi_df) if len(out_roi_df) > 0 else np.nan,
            "out_roi_path": str(out_roi_path),
            "out_z_path": str(out_z_path),
            "error_message": "",
        }

        return frame_metrics, file_log

    except Exception as e:
        return [], {
            **metadata,
            "status": "error",
            "error_message": str(e),
        }

In [ ]:
# from pathlib import Path
# import pandas as pd

# OUT_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/Preprocessing") / "Preprocessed_GC25_Z008"
# METRICS_DIR = OUT_BASE_DIR / "_metrics"

# frame_metrics_path = METRICS_DIR / "pre_dbscan_frame_metrics_z008.csv"
# file_logs_path = METRICS_DIR / "preprocessing_file_logs_z008.csv"

# frame_metrics_df = pd.read_csv(frame_metrics_path)
# file_logs_df = pd.read_csv(file_logs_path)

# print(frame_metrics_df.shape)
# print(file_logs_df.shape)
#=======================================================================================================
# Cell 7 - Menjalankan batch preprocessing seluruh dataset

existing_manifest = manifest_df[manifest_df["exists"]].copy()

all_frame_metrics = []
all_file_logs = []

print("===== START FULL DATASET PREPROCESSING =====")
print(f"Total existing CSV files to process: {len(existing_manifest):,}")
print(f"GC angle     : {GC_ANGLE_DEG} degree")
print(f"Z threshold  : {Z_GROUND_THRESHOLD} m")
print(f"Output ROI   : {GC25_ROI_DIR}")
print(f"Output Z     : {GC25_Z_DIR}")

for _, row in tqdm(existing_manifest.iterrows(), total=len(existing_manifest), desc="Processing CSV files"):
    frame_metrics, file_log = process_one_csv_file(row)

    if frame_metrics:
        all_frame_metrics.extend(frame_metrics)

    all_file_logs.append(file_log)

print("===== FINISHED PREPROCESSING =====")
print(f"Frame metric records : {len(all_frame_metrics):,}")
print(f"File log records     : {len(all_file_logs):,}")
#=======================================================================================================

# from pathlib import Path

# OUT_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/Preprocessing") / "Preprocessed_GC25_Z008"

# roi_files = list((OUT_BASE_DIR / "GC25_ROI").rglob("*.csv"))
# zt_files = list((OUT_BASE_DIR / "GC25_Z008").rglob("*.csv"))

# print("ROI files:", len(roi_files))
# print("ZT files :", len(zt_files))

In [ ]:
# Cell 8 - Simpan frame metrics dan file logs

frame_metrics_df = pd.DataFrame(all_frame_metrics)
file_logs_df = pd.DataFrame(all_file_logs)

frame_metrics_path = METRICS_DIR / "pre_dbscan_frame_metrics_z008.csv"
file_logs_path = METRICS_DIR / "preprocessing_file_logs_z008.csv"

frame_metrics_df.to_csv(frame_metrics_path, index=False, float_format=FLOAT_FORMAT)
file_logs_df.to_csv(file_logs_path, index=False, float_format=FLOAT_FORMAT)

print("===== SAVED METRICS =====")
print(f"Frame metrics saved : {frame_metrics_path}")
print(f"File logs saved     : {file_logs_path}")

print("\n===== FILE STATUS SUMMARY =====")
display(file_logs_df["status"].value_counts(dropna=False).reset_index().rename(
    columns={"index": "status", "status": "count"}
))

display(frame_metrics_df.head())
#=======================================================================================================
# Cell 8 Alternative - Load existing outputs and metrics safely

# from pathlib import Path
# import pandas as pd

# OUT_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/Preprocessing") / "Preprocessed_GC25_Z008"
# METRICS_DIR = OUT_BASE_DIR / "_metrics"

# GC25_ROI_DIR = OUT_BASE_DIR / "GC25_ROI"
# GC25_Z_DIR = OUT_BASE_DIR / "GC25_Z008"

# frame_metrics_path = METRICS_DIR / "pre_dbscan_frame_metrics_z008.csv"
# file_logs_path = METRICS_DIR / "preprocessing_file_logs_z008.csv"

# print("===== CHECK OUTPUT FILES =====")
# roi_files = list(GC25_ROI_DIR.rglob("*.csv")) if GC25_ROI_DIR.exists() else []
# zt_files = list(GC25_Z_DIR.rglob("*.csv")) if GC25_Z_DIR.exists() else []

# print("ROI dir exists:", GC25_ROI_DIR.exists())
# print("ZT dir exists :", GC25_Z_DIR.exists())
# print("ROI files:", len(roi_files))
# print("ZT files :", len(zt_files))

# print("\n===== CHECK METRICS FILES =====")
# print("Frame metrics exists:", frame_metrics_path.exists())
# print("File logs exists    :", file_logs_path.exists())

# if frame_metrics_path.exists() and file_logs_path.exists():
#     frame_metrics_df = pd.read_csv(frame_metrics_path)
#     file_logs_df = pd.read_csv(file_logs_path)

#     print("\n===== LOADED EXISTING METRICS =====")
#     print("frame_metrics_df:", frame_metrics_df.shape)
#     print("file_logs_df    :", file_logs_df.shape)

#     print("\n===== FILE STATUS SUMMARY =====")
#     display(
#         file_logs_df["status"]
#         .value_counts(dropna=False)
#         .reset_index()
#         .rename(columns={"index": "status", "status": "count"})
#     )

#     display(frame_metrics_df.head())

# else:
#     print("\nMetrics belum ditemukan.")
#     print("Kalau ROI/ZT files sudah lengkap, lanjutkan dengan notebook/cell diagnostik yang membaca langsung dari folder output.")

===== CHECK OUTPUT FILES =====
ROI dir exists: True
ZT dir exists : False
ROI files: 792
ZT files : 0

===== CHECK METRICS FILES =====
Frame metrics exists: False
File logs exists    : False

Metrics belum ditemukan.
Kalau ROI/ZT files sudah lengkap, lanjutkan dengan notebook/cell diagnostik yang membaca langsung dari folder output.


In [11]:
# Cell 9 - Summary per file, activity, room, subject, dan file_id

def summarize_group(df: pd.DataFrame, group_cols):
    """
    Membuat summary metrik pre-DBSCAN berdasarkan grup tertentu.
    """
    summary = df.groupby(group_cols).agg(
        n_frames=("frame_id", "count"),

        points_raw=("points_raw", "sum"),
        points_clean=("points_clean", "sum"),
        points_after_roi=("points_after_roi", "sum"),
        points_after_z_threshold=("points_after_z_threshold", "sum"),

        roi_retained_ratio_mean=("roi_retained_ratio", "mean"),
        retained_points_ratio_mean=("retained_points_ratio", "mean"),
        z_removed_ratio_mean=("z_removed_ratio_from_roi", "mean"),

        low_z_before_ratio_mean=("low_z_before_ratio", "mean"),
        low_z_residual_ratio_after_z_mean=("low_z_residual_ratio_after_z", "mean"),

        low_z_x_range_mean=("low_z_x_range", "mean"),
        low_z_y_range_mean=("low_z_y_range", "mean"),

        roi_z_p50_mean=("roi_z_p50", "mean"),
        roi_z_p95_mean=("roi_z_p95", "mean"),
        after_z_z_p50_mean=("after_z_z_p50", "mean"),
        after_z_z_p95_mean=("after_z_z_p95", "mean"),

        after_z_z_min_mean=("after_z_z_min", "mean"),
        after_z_z_max_mean=("after_z_z_max", "mean"),
    ).reset_index()

    # Weighted ratios berbasis jumlah titik
    summary["roi_retained_ratio_weighted"] = summary["points_after_roi"] / summary["points_clean"].replace(0, np.nan)
    summary["z_retained_ratio_weighted"] = summary["points_after_z_threshold"] / summary["points_after_roi"].replace(0, np.nan)
    summary["z_removed_ratio_weighted"] = 1.0 - summary["z_retained_ratio_weighted"]

    return summary


# Summary per source file
summary_by_file = summarize_group(
    frame_metrics_df,
    ["split", "room", "activity", "subject", "file_id"]
)

# Summary per activity
summary_by_activity = summarize_group(
    frame_metrics_df,
    ["activity"]
)

# Summary per split/room
summary_by_room = summarize_group(
    frame_metrics_df,
    ["split", "room"]
)

# Summary per subject
summary_by_subject = summarize_group(
    frame_metrics_df,
    ["split", "room", "subject"]
)

# Summary per 9 titik/file_id
summary_by_position = summarize_group(
    frame_metrics_df,
    ["file_id"]
)

# Save
summary_by_file_path = METRICS_DIR / "pre_dbscan_summary_by_file_z008.csv"
summary_by_activity_path = METRICS_DIR / "pre_dbscan_summary_by_activity_z008.csv"
summary_by_room_path = METRICS_DIR / "pre_dbscan_summary_by_room_z008.csv"
summary_by_subject_path = METRICS_DIR / "pre_dbscan_summary_by_subject_z008.csv"
summary_by_position_path = METRICS_DIR / "pre_dbscan_summary_by_position_z008.csv"

summary_by_file.to_csv(summary_by_file_path, index=False, float_format=FLOAT_FORMAT)
summary_by_activity.to_csv(summary_by_activity_path, index=False, float_format=FLOAT_FORMAT)
summary_by_room.to_csv(summary_by_room_path, index=False, float_format=FLOAT_FORMAT)
summary_by_subject.to_csv(summary_by_subject_path, index=False, float_format=FLOAT_FORMAT)
summary_by_position.to_csv(summary_by_position_path, index=False, float_format=FLOAT_FORMAT)

print("===== SUMMARY SAVED =====")
print(summary_by_file_path)
print(summary_by_activity_path)
print(summary_by_room_path)
print(summary_by_subject_path)
print(summary_by_position_path)

print("\n===== SUMMARY BY ACTIVITY =====")
display(summary_by_activity)

print("\n===== SUMMARY BY POSITION / FILE ID =====")
display(summary_by_position)

NameError: name 'frame_metrics_df' is not defined

In [13]:
# Cell 10 - Evaluasi global sebelum DBSCAN

global_summary = {
    "gc_angle_deg": GC_ANGLE_DEG,
    "z_ground_threshold": Z_GROUND_THRESHOLD,
    "n_files_success": int((file_logs_df["status"] == "success").sum()),
    "n_frames": int(len(frame_metrics_df)),

    "points_raw_total": int(frame_metrics_df["points_raw"].sum()),
    "points_clean_total": int(frame_metrics_df["points_clean"].sum()),
    "points_after_roi_total": int(frame_metrics_df["points_after_roi"].sum()),
    "points_after_z_threshold_total": int(frame_metrics_df["points_after_z_threshold"].sum()),
}

global_summary["roi_retained_ratio_weighted"] = (
    global_summary["points_after_roi_total"] / global_summary["points_clean_total"]
    if global_summary["points_clean_total"] > 0 else np.nan
)

global_summary["z_retained_ratio_weighted"] = (
    global_summary["points_after_z_threshold_total"] / global_summary["points_after_roi_total"]
    if global_summary["points_after_roi_total"] > 0 else np.nan
)

global_summary["z_removed_ratio_weighted"] = 1.0 - global_summary["z_retained_ratio_weighted"]

global_summary["low_z_before_ratio_mean"] = float(frame_metrics_df["low_z_before_ratio"].mean())
global_summary["low_z_residual_ratio_after_z_mean"] = float(frame_metrics_df["low_z_residual_ratio_after_z"].mean())

global_summary["low_z_x_range_mean"] = float(frame_metrics_df["low_z_x_range"].mean())
global_summary["low_z_y_range_mean"] = float(frame_metrics_df["low_z_y_range"].mean())

global_summary_df = pd.DataFrame([global_summary])

global_summary_path = METRICS_DIR / "pre_dbscan_global_summary_z008.csv"
global_summary_df.to_csv(global_summary_path, index=False, float_format=FLOAT_FORMAT)

print("===== GLOBAL SUMMARY BEFORE DBSCAN =====")
display(global_summary_df)

print(f"Global summary saved: {global_summary_path}")

NameError: name 'file_logs_df' is not defined

In [14]:
# Cell 11 - Quick sanity check output file

# Ambil satu file sukses untuk dicek
success_files = file_logs_df[file_logs_df["status"] == "success"].copy()

if len(success_files) > 0:
    sample_row = success_files.iloc[0]

    sample_roi_path = Path(sample_row["out_roi_path"])
    sample_z_path = Path(sample_row["out_z_path"])

    print("===== SAMPLE OUTPUT CHECK =====")
    print(f"Sample ROI file : {sample_roi_path}")
    print(f"Sample Z file   : {sample_z_path}")

    sample_roi_df = pd.read_csv(sample_roi_path)
    sample_z_df = pd.read_csv(sample_z_path)

    print("\nROI output shape:", sample_roi_df.shape)
    print("Z output shape  :", sample_z_df.shape)

    print("\nROI columns:")
    print(sample_roi_df.columns.tolist())

    print("\nZ-threshold output columns:")
    print(sample_z_df.columns.tolist())

    print("\nROI coordinate range:")
    display(sample_roi_df[["X_corr", "Y_corr", "Z_level"]].describe())

    print("\nZ-threshold coordinate range:")
    display(sample_z_df[["X_corr", "Y_corr", "Z_level"]].describe())

else:
    print("No successful files found.")

NameError: name 'file_logs_df' is not defined

In [15]:
# Cell 12 - Optional plot distribusi Z sebelum DBSCAN

import matplotlib.pyplot as plt

# Plot histogram Z_level dari sample file
if len(success_files) > 0:
    sample_roi_df = pd.read_csv(sample_roi_path)
    sample_z_df = pd.read_csv(sample_z_path)

    plt.figure(figsize=(9, 5))
    plt.hist(sample_roi_df["Z_level"], bins=100, alpha=0.7, label="After GC + ROI")
    plt.axvline(Z_GROUND_THRESHOLD, linestyle="--", label=f"Z threshold = {Z_GROUND_THRESHOLD:.2f} m")
    plt.axvline(LOW_Z_RESIDUAL_LIMIT, linestyle="--", label=f"Low-Z residual limit = {LOW_Z_RESIDUAL_LIMIT:.2f} m")
    plt.xlabel("Z_level (m)")
    plt.ylabel("Number of points")
    plt.title("Height Distribution After GC + ROI")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    fig_path = OUT_BASE_DIR / "_analysis_z_distribution_sample.png"
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print(f"Figure saved: {fig_path}")

NameError: name 'success_files' is not defined

In [16]:
# Cell 13 - Optional cek file error

error_logs = file_logs_df[file_logs_df["status"] != "success"].copy()

print("===== NON-SUCCESS FILES =====")
print(f"Count: {len(error_logs)}")

if len(error_logs) > 0:
    display(error_logs[["split", "room", "activity", "subject", "file_id", "status", "error_message", "source_csv"]].head(30))

    error_path = LOG_DIR / "non_success_files_z008.csv"
    error_logs.to_csv(error_path, index=False)
    print(f"Non-success logs saved: {error_path}")
else:
    print("All existing files processed successfully.")

NameError: name 'file_logs_df' is not defined